# 📰 Automatic Headline Generator

##Git Config & Auth

In [21]:
import os
from google.colab import userdata
# Configure git identity
!git config --global user.email "alizazia2005@gmail.com"
!git config --global user.name "moonlightaliza"

# Clone your repo (only run once)
!git clone https://github.com/moonlightaliza/headline-generation-nlp.git

# Move into repo folder
os.chdir('/content/headline-generation-nlp')
print("Working directory:", os.getcwd())

Cloning into 'headline-generation-nlp'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), done.
Working directory: /content/headline-generation-nlp


In [22]:
import os
print(os.getcwd())
!ls

/content/headline-generation-nlp
README.md


In [23]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [26]:
!find /content/drive/MyDrive -name "*.ipynb" 2>/dev/null
!ls /content/headline-generation-nlp

README.md


In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

##1. Installations

In [1]:
!pip install tensorflow kagglehub rouge_score nltk --quiet

  Preparing metadata (setup.py) ... done


##2. Import Libraries

In [2]:
import kagglehub
import pandas as pd
import numpy as np
import os
import re
import nltk
nltk.download('stopwords')

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


##3. Load Dataset

In [4]:
path = kagglehub.dataset_download("gowrishankarp/newspaper-text-summarization-cnn-dailymail")

print("Path:", path)

Using Colab cache for faster access to the 'newspaper-text-summarization-cnn-dailymail' dataset.
Path: /kaggle/input/newspaper-text-summarization-cnn-dailymail


In [5]:
# Check files
print(os.listdir(path))

['cnn_dailymail']


In [6]:
inner_path = os.path.join(path, "cnn_dailymail")
print(os.listdir(inner_path))

['validation.csv', 'train.csv', 'test.csv']


In [7]:
file_path = os.path.join(inner_path, "train.csv")
df = pd.read_csv(file_path)


In [8]:
df.head()

,id,article,highlights
0,0001d1afc246a7964130f43ae940af6bc6c57f01,By . Associated Press . PUBLISHED: . 14:11 EST...,"Bishop John Folda, of North Dakota, is taking ..."
1,0002095e55fcbd3a2f366d9bf92a95433dc305ef,(CNN) -- Ralph Mata was an internal affairs li...,Criminal complaint: Cop used his role to help ...
2,00027e965c8264c35cc1bc55556db388da82b07f,A drunk driver who killed a young woman in a h...,"Craig Eccleston-Todd, 27, had drunk at least t..."
3,0002c17436637c4fe1837c935c04de47adb18e9a,(CNN) -- With a breezy sweep of his pen Presid...,Nina dos Santos says Europe must be ready to a...
4,0003ad6ef0c37534f80b55b4235108024b407f0b,Fleetwood are the only team still to have a 10...,Fleetwood top of League One after 2-0 win at S...


In [9]:
print(df.columns)

Index(['id', 'article', 'highlights'], dtype='object')


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 287113 entries, 0 to 287112
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   id          287113 non-null  object
 1   article     287113 non-null  object
 2   highlights  287113 non-null  object
dtypes: object(3)
memory usage: 6.6+ MB


##4. Preprocessing

###4.1 Reducing Dataset

In [11]:
df = df[['article', 'highlights']].dropna()

# take small subset
df = df.sample(5000, random_state=42)

In [12]:
df.head()

,article,highlights
272581,By . Mia De Graaf . Britons flocked to beaches...,People enjoyed temperatures of 17C at Brighton...
772,A couple who weighed a combined 32st were sham...,Couple started piling on pounds after the birt...
171868,Video footage shows the heart stopping moment ...,A 17-year-old boy suffering lacerations to his...
63167,"Istanbul, Turkey (CNN) -- About 250 people rac...",Syrians citizens hightail it to Turkey .\nMost...
68522,By . Daily Mail Reporter . PUBLISHED: . 12:53 ...,The Xue Long had provided the helicopter that ...


### 4.2 Cleaning Dataset

In [19]:
def clean_text(text):
    text = text.lower()
    #CNN/DailyMail articles often start with "(CNN)" as a dateline tag
    #Remove the tag
    text = re.sub(r'\(cnn\)', '', text)
    #Replace em dashes with space eg. war--peace -> war peace
    text = re.sub(r'--', ' ', text)
    #Replace he's like contractions to he is
    text = re.sub(r"\'s", " is", text)
    #Replace wasn't like contractions to was not
    text = re.sub(r"n\'t", " not", text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)   # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()       # remove extra spaces
    return text

df['article_clean'] = df['article'].apply(clean_text)

In [20]:
df['article_clean']

,article_clean
272581,by mia de graaf britons flocked to beaches acr...
772,a couple who weighed a combined 32st were sham...
171868,video footage shows the heart stopping moment ...
63167,istanbul turkey about 250 people raced across ...
68522,by daily mail reporter published 1253 est 3 ja...
...,...
271171,by matt chorley mailonline political editor pu...
146080,st poelten austria a verdict in the case of jo...
270020,by hugo gye published 0749 est 22 january 2014...
126659,it is no super bowl heck it is no monday night...
